In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
import xgboost as xgb

from sklearn.base import clone
from sklearn.calibration import (
    CalibratedClassifierCV,
    calibration_curve
)
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    fbeta_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_predict,
    train_test_split
)

pd.set_option("display.max_columns", None)

print("Scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgb.__version__)

In [ ]:
def find_project_root(start_path: Path) -> Path:
    """
    Find the project folder containing the
    processed customer-churn dataset.
    """

    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:

        expected_dataset = (
            candidate
            / "data"
            / "processed"
            / "telco_churn_model.csv"
        )

        if expected_dataset.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find "
        "data/processed/telco_churn_model.csv. "
        "Open this notebook from inside the "
        "customer-churn-project folder."
    )


project_root = find_project_root(Path.cwd())

models_directory = (
    project_root
    / "models"
)

model_output_directory = (
    project_root
    / "reports"
    / "model_results"
)

figures_directory = (
    project_root
    / "reports"
    / "figures"
)

models_directory.mkdir(
    parents=True,
    exist_ok=True
)

model_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

figures_directory.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", project_root)

In [ ]:
model_data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_model.csv"
)

model_df = pd.read_csv(
    model_data_path
)

print("Dataset path:", model_data_path)
print("Dataset shape:", model_df.shape)

display(model_df.head())

In [ ]:
target_column = "churn_value"

forbidden_columns = {
    "customerid",
    "churn_label",
    "churn_score",
    "churn_reason"
}

assert target_column in model_df.columns

assert model_df[target_column].isin(
    [0, 1]
).all()

assert (
    model_df[target_column]
    .isna()
    .sum()
    == 0
)

assert not forbidden_columns.intersection(
    model_df.columns
), (
    "Identifier or leakage columns found: "
    f"{forbidden_columns.intersection(model_df.columns)}"
)


X = model_df.drop(
    columns=[target_column]
).copy()

y = model_df[
    target_column
].astype(int).copy()


print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Overall churn rate:", round(y.mean(), 4))

In [ ]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print(
    "Training churn rate:",
    round(y_train.mean(), 4)
)

print(
    "Test churn rate:",
    round(y_test.mean(), 4)
)

In [ ]:
best_tuned_model_path = (
    models_directory
    / "best_tuned_cv_model.joblib"
)

if not best_tuned_model_path.exists():

    raise FileNotFoundError(
        "best_tuned_cv_model.joblib was not found. "
        "Complete Notebook 4 first."
    )


best_tuned_model = joblib.load(
    best_tuned_model_path
)


print("Loaded model type:")
print(type(best_tuned_model))

print("\nPipeline steps:")
print(best_tuned_model.named_steps.keys())

In [ ]:
tuning_metadata_path = (
    models_directory
    / "hyperparameter_tuning_metadata.json"
)


if tuning_metadata_path.exists():

    with open(
        tuning_metadata_path,
        "r",
        encoding="utf-8"
    ) as file:

        tuning_metadata = json.load(file)

else:

    tuning_metadata = {}


selected_tuned_model_name = (
    tuning_metadata.get(
        "selected_model",
        type(
            best_tuned_model.named_steps[
                "classifier"
            ]
        ).__name__
    )
)


print(
    "Selected tuned model:",
    selected_tuned_model_name
)

print(
    "Selection metric:",
    tuning_metadata.get(
        "primary_selection_metric",
        "Cross-validation PR-AUC"
    )
)

In [ ]:
outer_cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print(
    "Outer cross-validation folds:",
    outer_cross_validation.n_splits
)

In [ ]:
uncalibrated_oof_probabilities = (
    cross_val_predict(
        estimator=clone(best_tuned_model),
        X=X_train,
        y=y_train,
        cv=outer_cross_validation,
        method="predict_proba",

        # Keep sequential for GPU compatibility
        n_jobs=1
    )[:, 1]
)


assert len(
    uncalibrated_oof_probabilities
) == len(y_train)

assert np.all(
    (
        uncalibrated_oof_probabilities
        >= 0
    )
    &
    (
        uncalibrated_oof_probabilities
        <= 1
    )
)


print(
    "Generated uncalibrated OOF probabilities:",
    len(uncalibrated_oof_probabilities)
)

print(
    "Minimum probability:",
    round(
        uncalibrated_oof_probabilities.min(),
        4
    )
)

print(
    "Maximum probability:",
    round(
        uncalibrated_oof_probabilities.max(),
        4
    )
)

In [ ]:
calibration_inner_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=43
)


calibrated_model_template = (
    CalibratedClassifierCV(
        estimator=clone(best_tuned_model),

        method="sigmoid",

        cv=calibration_inner_cv,

        # Avoid multiple simultaneous GPU fits
        n_jobs=1,

        ensemble="auto"
    )
)


print(
    "Sigmoid calibration template created."
)

In [ ]:
calibrated_oof_probabilities = (
    cross_val_predict(
        estimator=calibrated_model_template,
        X=X_train,
        y=y_train,
        cv=outer_cross_validation,
        method="predict_proba",
        n_jobs=1
    )[:, 1]
)


assert len(
    calibrated_oof_probabilities
) == len(y_train)

assert np.all(
    (
        calibrated_oof_probabilities >= 0
    )
    &
    (
        calibrated_oof_probabilities <= 1
    )
)


print(
    "Generated calibrated OOF probabilities:",
    len(calibrated_oof_probabilities)
)

In [ ]:
def evaluate_probability_quality(
    model_label,
    y_true,
    probabilities
):
    """
    Evaluate ranking and calibration quality.
    """

    probabilities = np.asarray(
        probabilities
    )

    probabilities = np.clip(
        probabilities,
        1e-8,
        1 - 1e-8
    )

    two_class_probabilities = (
        np.column_stack(
            [
                1 - probabilities,
                probabilities
            ]
        )
    )

    return {
        "probability_model": model_label,

        "roc_auc": roc_auc_score(
            y_true,
            probabilities
        ),

        "pr_auc": average_precision_score(
            y_true,
            probabilities
        ),

        "brier_loss": brier_score_loss(
            y_true,
            probabilities
        ),

        "log_loss": log_loss(
            y_true,
            two_class_probabilities,
            labels=[0, 1]
        ),

        "mean_predicted_probability": (
            probabilities.mean()
        ),

        "actual_churn_rate": (
            np.asarray(y_true).mean()
        )
    }

In [ ]:
probability_quality_comparison = pd.DataFrame(
    [
        evaluate_probability_quality(
            model_label="Uncalibrated",
            y_true=y_train,
            probabilities=(
                uncalibrated_oof_probabilities
            )
        ),

        evaluate_probability_quality(
            model_label="Sigmoid calibrated",
            y_true=y_train,
            probabilities=(
                calibrated_oof_probabilities
            )
        )
    ]
)


display(
    probability_quality_comparison.round(4)
)

In [ ]:
uncalibrated_brier = float(
    probability_quality_comparison.loc[
        probability_quality_comparison[
            "probability_model"
        ] == "Uncalibrated",
        "brier_loss"
    ].iloc[0]
)


calibrated_brier = float(
    probability_quality_comparison.loc[
        probability_quality_comparison[
            "probability_model"
        ] == "Sigmoid calibrated",
        "brier_loss"
    ].iloc[0]
)


minimum_brier_improvement = 0.001


if (
    calibrated_brier
    <=
    uncalibrated_brier
    -
    minimum_brier_improvement
):

    selected_probability_source = (
        "Sigmoid calibrated"
    )

    selected_oof_probabilities = (
        calibrated_oof_probabilities
    )

else:

    selected_probability_source = (
        "Uncalibrated"
    )

    selected_oof_probabilities = (
        uncalibrated_oof_probabilities
    )


print(
    "Selected probability source:",
    selected_probability_source
)

print(
    "Uncalibrated Brier loss:",
    round(uncalibrated_brier, 4)
)

print(
    "Calibrated Brier loss:",
    round(calibrated_brier, 4)
)

In [ ]:
uncalibrated_fraction_positive, \
uncalibrated_mean_probability = (
    calibration_curve(
        y_train,
        uncalibrated_oof_probabilities,
        n_bins=10,
        strategy="quantile"
    )
)


calibrated_fraction_positive, \
calibrated_mean_probability = (
    calibration_curve(
        y_train,
        calibrated_oof_probabilities,
        n_bins=10,
        strategy="quantile"
    )
)


plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    uncalibrated_mean_probability,
    uncalibrated_fraction_positive,
    marker="o",
    label="Uncalibrated"
)

plt.plot(
    calibrated_mean_probability,
    calibrated_fraction_positive,
    marker="o",
    label="Sigmoid calibrated"
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Churn Rate")

plt.title(
    "OOF Probability Calibration Comparison"
)

plt.legend()
plt.tight_layout()

calibration_figure_path = (
    figures_directory
    / "oof_calibration_comparison.png"
)

plt.savefig(
    calibration_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def evaluate_thresholds(
    y_true,
    probabilities,
    thresholds=None,
    false_negative_cost=5,
    false_positive_cost=1
):
    """
    Evaluate classification metrics across thresholds.

    Cost values are illustrative unless supported
    by actual business estimates.
    """

    if thresholds is None:

        thresholds = np.round(
            np.arange(
                0.05,
                0.951,
                0.01
            ),
            2
        )

    rows = []

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true,
            predictions,
            labels=[0, 1]
        ).ravel()

        rows.append(
            {
                "threshold": threshold,

                "accuracy": accuracy_score(
                    y_true,
                    predictions
                ),

                "balanced_accuracy": (
                    balanced_accuracy_score(
                        y_true,
                        predictions
                    )
                ),

                "precision": precision_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),

                "recall": recall_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),

                "f1": f1_score(
                    y_true,
                    predictions,
                    zero_division=0
                ),

                "f2": fbeta_score(
                    y_true,
                    predictions,
                    beta=2,
                    zero_division=0
                ),

                "true_negatives": tn,
                "false_positives": fp,
                "false_negatives": fn,
                "true_positives": tp,

                "customers_flagged": int(
                    predictions.sum()
                ),

                "estimated_cost": (
                    fn * false_negative_cost
                    +
                    fp * false_positive_cost
                )
            }
        )

    return pd.DataFrame(rows)

In [ ]:
FALSE_NEGATIVE_COST = 5
FALSE_POSITIVE_COST = 1


threshold_results = evaluate_thresholds(
    y_true=y_train,
    probabilities=selected_oof_probabilities,

    false_negative_cost=(
        FALSE_NEGATIVE_COST
    ),

    false_positive_cost=(
        FALSE_POSITIVE_COST
    )
)


display(
    threshold_results.head().round(4)
)

In [ ]:
best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]


best_f2_row = threshold_results.loc[
    threshold_results["f2"].idxmax()
]


lowest_cost_row = threshold_results.loc[
    threshold_results[
        "estimated_cost"
    ].idxmin()
]


candidate_thresholds = pd.DataFrame(
    [
        {
            "selection_method": (
                "Default threshold"
            ),
            "threshold": 0.50
        },

        {
            "selection_method": (
                "Maximum F1"
            ),
            "threshold": float(
                best_f1_row["threshold"]
            )
        },

        {
            "selection_method": (
                "Maximum F2"
            ),
            "threshold": float(
                best_f2_row["threshold"]
            )
        },

        {
            "selection_method": (
                "Minimum illustrative cost"
            ),
            "threshold": float(
                lowest_cost_row["threshold"]
            )
        }
    ]
)


display(candidate_thresholds)

In [ ]:
THRESHOLD_OBJECTIVE = "f2"

# Valid options:
# "f1"
# "f2"
# "estimated_cost"


if THRESHOLD_OBJECTIVE in {
    "f1",
    "f2"
}:

    selected_threshold_row = (
        threshold_results.loc[
            threshold_results[
                THRESHOLD_OBJECTIVE
            ].idxmax()
        ]
    )

elif THRESHOLD_OBJECTIVE == "estimated_cost":

    selected_threshold_row = (
        threshold_results.loc[
            threshold_results[
                "estimated_cost"
            ].idxmin()
        ]
    )

else:

    raise ValueError(
        "THRESHOLD_OBJECTIVE must be "
        "'f1', 'f2', or 'estimated_cost'."
    )


selected_threshold = float(
    selected_threshold_row["threshold"]
)


print(
    "Threshold objective:",
    THRESHOLD_OBJECTIVE
)

print(
    "Selected threshold:",
    round(selected_threshold, 3)
)

display(
    selected_threshold_row
    .to_frame(name="value")
    .round(4)
)

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    label="Precision"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    label="Recall"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    label="F1"
)

plt.plot(
    threshold_results["threshold"],
    threshold_results["f2"],
    label="F2"
)

plt.axvline(
    selected_threshold,
    linestyle="--",
    label=(
        f"Selected threshold = "
        f"{selected_threshold:.2f}"
    )
)

plt.axvline(
    0.50,
    linestyle=":",
    label="Default threshold = 0.50"
)

plt.xlabel("Probability Threshold")
plt.ylabel("Metric Score")

plt.title(
    "OOF Threshold Performance"
)

plt.ylim(0, 1)
plt.legend()

plt.tight_layout()

threshold_figure_path = (
    figures_directory
    / "threshold_performance.png"
)

plt.savefig(
    threshold_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
def classification_metrics_at_threshold(
    model_label,
    y_true,
    probabilities,
    threshold
):
    """
    Calculate classification metrics at one threshold.
    """

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "model": model_label,

        "threshold": threshold,

        "accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true,
                predictions
            )
        ),

        "precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "f2": fbeta_score(
            y_true,
            predictions,
            beta=2,
            zero_division=0
        ),

        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp
    }

In [ ]:
oof_threshold_comparison = pd.DataFrame(
    [
        classification_metrics_at_threshold(
            model_label="Default threshold",
            y_true=y_train,
            probabilities=(
                selected_oof_probabilities
            ),
            threshold=0.50
        ),

        classification_metrics_at_threshold(
            model_label="Tuned threshold",
            y_true=y_train,
            probabilities=(
                selected_oof_probabilities
            ),
            threshold=selected_threshold
        )
    ]
)


display(
    oof_threshold_comparison.round(4)
)

In [ ]:
if (
    selected_probability_source
    == "Sigmoid calibrated"
):

    final_model = clone(
        calibrated_model_template
    )

else:

    final_model = clone(
        best_tuned_model
    )


print(
    "Final model probability type:",
    selected_probability_source
)

In [ ]:
final_model.fit(
    X_train,
    y_train
)


print(
    "Final tuned model fitted on "
    "the complete training set."
)

In [ ]:
test_probabilities = (
    final_model.predict_proba(
        X_test
    )[:, 1]
)


default_test_predictions = (
    test_probabilities >= 0.50
).astype(int)


tuned_test_predictions = (
    test_probabilities
    >=
    selected_threshold
).astype(int)


assert len(test_probabilities) == len(y_test)

print(
    "Generated test probabilities:",
    len(test_probabilities)
)

In [ ]:
test_probability_metrics = pd.DataFrame(
    [
        evaluate_probability_quality(
            model_label=(
                selected_probability_source
            ),
            y_true=y_test,
            probabilities=test_probabilities
        )
    ]
)


display(
    test_probability_metrics.round(4)
)

In [ ]:
test_threshold_comparison = pd.DataFrame(
    [
        classification_metrics_at_threshold(
            model_label=(
                f"{selected_tuned_model_name} "
                "— Default"
            ),
            y_true=y_test,
            probabilities=test_probabilities,
            threshold=0.50
        ),

        classification_metrics_at_threshold(
            model_label=(
                f"{selected_tuned_model_name} "
                "— Tuned"
            ),
            y_true=y_test,
            probabilities=test_probabilities,
            threshold=selected_threshold
        )
    ]
)


test_roc_auc = roc_auc_score(
    y_test,
    test_probabilities
)

test_pr_auc = average_precision_score(
    y_test,
    test_probabilities
)

test_brier_loss = brier_score_loss(
    y_test,
    test_probabilities
)


test_threshold_comparison[
    "roc_auc"
] = test_roc_auc

test_threshold_comparison[
    "pr_auc"
] = test_pr_auc

test_threshold_comparison[
    "brier_loss"
] = test_brier_loss


display(
    test_threshold_comparison.round(4)
)

In [ ]:
print(
    "Final selected model:",
    selected_tuned_model_name
)

print(
    "Probability source:",
    selected_probability_source
)

print(
    "Selected threshold:",
    round(selected_threshold, 3)
)

print("\nClassification report:\n")


print(
    classification_report(
        y_test,
        tuned_test_predictions,

        target_names=[
            "Not Churned",
            "Churned"
        ],

        digits=4,
        zero_division=0
    )
)

In [ ]:
test_confusion_matrix = confusion_matrix(
    y_test,
    tuned_test_predictions,
    labels=[0, 1]
)


confusion_display = ConfusionMatrixDisplay(
    confusion_matrix=test_confusion_matrix,

    display_labels=[
        "Not Churned",
        "Churned"
    ]
)


confusion_display.plot(
    values_format="d"
)


plt.title(
    f"Final Test Confusion Matrix\n"
    f"Threshold = {selected_threshold:.2f}"
)

plt.tight_layout()


confusion_figure_path = (
    figures_directory
    / "final_test_confusion_matrix.png"
)

plt.savefig(
    confusion_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
false_positive_rate, \
true_positive_rate, \
roc_thresholds = roc_curve(
    y_test,
    test_probabilities
)


plt.figure(figsize=(8, 6))

plt.plot(
    false_positive_rate,
    true_positive_rate,
    label=(
        f"ROC-AUC = "
        f"{test_roc_auc:.3f}"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Random classifier"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("Final Test ROC Curve")

plt.legend()
plt.tight_layout()


roc_figure_path = (
    figures_directory
    / "final_test_roc_curve.png"
)

plt.savefig(
    roc_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
test_precision_curve, \
test_recall_curve, \
pr_thresholds = precision_recall_curve(
    y_test,
    test_probabilities
)


test_churn_prevalence = y_test.mean()


plt.figure(figsize=(8, 6))

plt.plot(
    test_recall_curve,
    test_precision_curve,
    label=(
        f"PR-AUC = "
        f"{test_pr_auc:.3f}"
    )
)

plt.axhline(
    test_churn_prevalence,
    linestyle="--",
    label=(
        f"Random baseline = "
        f"{test_churn_prevalence:.3f}"
    )
)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title(
    "Final Test Precision–Recall Curve"
)

plt.legend()
plt.tight_layout()


pr_figure_path = (
    figures_directory
    / "final_test_precision_recall_curve.png"
)

plt.savefig(
    pr_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
test_fraction_positive, \
test_mean_probability = calibration_curve(
    y_test,
    test_probabilities,
    n_bins=10,
    strategy="quantile"
)


plt.figure(figsize=(8, 6))

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Perfect calibration"
)

plt.plot(
    test_mean_probability,
    test_fraction_positive,
    marker="o",
    label=selected_probability_source
)

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Churn Rate")

plt.title(
    "Final Test Calibration Curve"
)

plt.legend()
plt.tight_layout()


test_calibration_figure_path = (
    figures_directory
    / "final_test_calibration_curve.png"
)

plt.savefig(
    test_calibration_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
enriched_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics_enriched.csv"
)


base_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics.csv"
)


if enriched_analytics_path.exists():

    analytics_lookup = pd.read_csv(
        enriched_analytics_path
    )

else:

    analytics_lookup = pd.read_csv(
        base_analytics_path
    )


assert len(analytics_lookup) == len(model_df)

assert "customerid" in analytics_lookup.columns
assert "cltv" in analytics_lookup.columns


print(
    "Analytics lookup shape:",
    analytics_lookup.shape
)

In [ ]:
high_risk_cutoff = min(
    0.95,
    max(
        0.60,
        selected_threshold + 0.20
    )
)


if high_risk_cutoff <= selected_threshold:

    high_risk_cutoff = min(
        0.99,
        selected_threshold + 0.05
    )


print(
    "Low-risk upper limit:",
    round(selected_threshold, 3)
)

print(
    "High-risk lower limit:",
    round(high_risk_cutoff, 3)
)

In [ ]:
customer_columns = [
    "customerid",
    "cltv",
    "monthly_charges",
    "total_charges",
    "contract",
    "tenure_months",
    "internet_service",
    "payment_method",
    "city"
]


available_customer_columns = [
    column
    for column in customer_columns
    if column in analytics_lookup.columns
]


test_customer_predictions = (
    analytics_lookup.loc[
        X_test.index,
        available_customer_columns
    ]
    .copy()
)


test_customer_predictions[
    "actual_churn"
] = y_test.loc[
    X_test.index
].values


test_customer_predictions[
    "churn_probability"
] = test_probabilities


test_customer_predictions[
    "predicted_churn"
] = tuned_test_predictions


test_customer_predictions[
    "selected_threshold"
] = selected_threshold


test_customer_predictions[
    "risk_segment"
] = pd.cut(
    test_customer_predictions[
        "churn_probability"
    ],

    bins=[
        -np.inf,
        selected_threshold,
        high_risk_cutoff,
        np.inf
    ],

    labels=[
        "Low Risk",
        "Elevated Risk",
        "High Risk"
    ],

    right=False
)


test_customer_predictions[
    "priority_score"
] = (
    test_customer_predictions[
        "churn_probability"
    ]
    *
    test_customer_predictions[
        "cltv"
    ]
)


display(
    test_customer_predictions
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

In [ ]:
all_out_of_sample_probabilities = pd.Series(
    index=model_df.index,
    dtype=float
)


all_out_of_sample_probabilities.loc[
    X_train.index
] = selected_oof_probabilities


all_out_of_sample_probabilities.loc[
    X_test.index
] = test_probabilities


assert (
    all_out_of_sample_probabilities
    .isna()
    .sum()
    == 0
)


all_prediction_sources = pd.Series(
    index=model_df.index,
    dtype="object"
)


all_prediction_sources.loc[
    X_train.index
] = "Training OOF"


all_prediction_sources.loc[
    X_test.index
] = "Test holdout"


print(
    "All out-of-sample probabilities created:",
    len(all_out_of_sample_probabilities)
)

In [ ]:
all_customer_predictions = (
    analytics_lookup.copy()
)


all_customer_predictions[
    "actual_churn"
] = y.values


all_customer_predictions[
    "churn_probability"
] = (
    all_out_of_sample_probabilities
    .sort_index()
    .values
)


all_customer_predictions[
    "prediction_source"
] = (
    all_prediction_sources
    .sort_index()
    .values
)


all_customer_predictions[
    "predicted_churn"
] = (
    all_customer_predictions[
        "churn_probability"
    ]
    >=
    selected_threshold
).astype(int)


all_customer_predictions[
    "selected_threshold"
] = selected_threshold


all_customer_predictions[
    "risk_segment"
] = pd.cut(
    all_customer_predictions[
        "churn_probability"
    ],

    bins=[
        -np.inf,
        selected_threshold,
        high_risk_cutoff,
        np.inf
    ],

    labels=[
        "Low Risk",
        "Elevated Risk",
        "High Risk"
    ],

    right=False
)


all_customer_predictions[
    "priority_score"
] = (
    all_customer_predictions[
        "churn_probability"
    ]
    *
    all_customer_predictions[
        "cltv"
    ]
)


print(
    "Full prediction table shape:",
    all_customer_predictions.shape
)


display(
    all_customer_predictions[
        [
            "customerid",
            "actual_churn",
            "churn_probability",
            "predicted_churn",
            "risk_segment",
            "cltv",
            "priority_score",
            "prediction_source"
        ]
    ]
    .sort_values(
        "priority_score",
        ascending=False
    )
    .head(20)
)

In [ ]:
final_evaluation_model_path = (
    models_directory
    / "final_tuned_churn_model.joblib"
)


joblib.dump(
    final_model,
    final_evaluation_model_path
)


print(
    "Final evaluation model saved to:"
)

print(
    final_evaluation_model_path
)

In [ ]:
probability_quality_comparison.to_csv(
    model_output_directory
    / "probability_calibration_comparison.csv",
    index=False
)


threshold_results.to_csv(
    model_output_directory
    / "final_threshold_analysis.csv",
    index=False
)


candidate_thresholds.to_csv(
    model_output_directory
    / "candidate_thresholds.csv",
    index=False
)


oof_threshold_comparison.to_csv(
    model_output_directory
    / "final_oof_threshold_comparison.csv",
    index=False
)


test_probability_metrics.to_csv(
    model_output_directory
    / "final_test_probability_metrics.csv",
    index=False
)


test_threshold_comparison.to_csv(
    model_output_directory
    / "final_test_threshold_comparison.csv",
    index=False
)


test_customer_predictions.to_csv(
    model_output_directory
    / "final_test_customer_predictions.csv",
    index=False
)


all_customer_predictions.to_csv(
    model_output_directory
    / "all_customer_oos_predictions.csv",
    index=False
)


print(
    "Final model results saved to:"
)

print(
    model_output_directory
)

In [ ]:
final_model_metadata = {
    "notebook": "05_final_tuned_model",

    "model_name": selected_tuned_model_name,

    "target": target_column,

    "probability_source": (
        selected_probability_source
    ),

    "calibration_method": (
        "sigmoid"
        if selected_probability_source
        == "Sigmoid calibrated"
        else None
    ),

    "threshold_selection_objective": (
        THRESHOLD_OBJECTIVE
    ),

    "selected_threshold": (
        selected_threshold
    ),

    "high_risk_cutoff": (
        high_risk_cutoff
    ),

    "test_metrics": {
        "roc_auc": float(test_roc_auc),

        "pr_auc": float(test_pr_auc),

        "brier_loss": float(
            test_brier_loss
        ),

        "precision": float(
            precision_score(
                y_test,
                tuned_test_predictions,
                zero_division=0
            )
        ),

        "recall": float(
            recall_score(
                y_test,
                tuned_test_predictions,
                zero_division=0
            )
        ),

        "f1": float(
            f1_score(
                y_test,
                tuned_test_predictions,
                zero_division=0
            )
        ),

        "f2": float(
            fbeta_score(
                y_test,
                tuned_test_predictions,
                beta=2,
                zero_division=0
            )
        )
    },

    "training_rows": int(
        X_train.shape[0]
    ),

    "test_rows": int(
        X_test.shape[0]
    ),

    "random_state": 42,

    "scikit_learn_version": (
        sklearn.__version__
    ),

    "xgboost_version": (
        xgb.__version__
    )
}


final_metadata_path = (
    models_directory
    / "final_tuned_churn_model_metadata.json"
)


with open(
    final_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_model_metadata,
        file,
        indent=4
    )


print(
    "Final metadata saved to:"
)

print(
    final_metadata_path
)

In [ ]:
if (
    selected_probability_source
    == "Sigmoid calibrated"
):

    production_model = clone(
        calibrated_model_template
    )

else:

    production_model = clone(
        best_tuned_model
    )


production_model.fit(
    X,
    y
)


print(
    "Production model trained on "
    "all labelled customers."
)

In [ ]:
production_model_path = (
    models_directory
    / "production_churn_model.joblib"
)


joblib.dump(
    production_model,
    production_model_path
)


production_metadata = {
    "model_name": selected_tuned_model_name,

    "training_rows": int(len(X)),

    "probability_source": (
        selected_probability_source
    ),

    "selected_threshold": (
        selected_threshold
    ),

    "high_risk_cutoff": (
        high_risk_cutoff
    ),

    "purpose": (
        "Future customer scoring and deployment"
    ),

    "evaluation_metrics_source": (
        "final_tuned_churn_model.joblib "
        "and final holdout evaluation"
    )
}


production_metadata_path = (
    models_directory
    / "production_churn_model_metadata.json"
)


with open(
    production_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        production_metadata,
        file,
        indent=4
    )


print(
    "Production model saved to:"
)

print(
    production_model_path
)

In [ ]:
required_artifacts = [
    final_evaluation_model_path,

    final_metadata_path,

    production_model_path,

    production_metadata_path,

    model_output_directory
    / "final_threshold_analysis.csv",

    model_output_directory
    / "final_test_threshold_comparison.csv",

    model_output_directory
    / "final_test_customer_predictions.csv",

    model_output_directory
    / "all_customer_oos_predictions.csv",

    figures_directory
    / "final_test_confusion_matrix.png",

    figures_directory
    / "final_test_roc_curve.png",

    figures_directory
    / "final_test_precision_recall_curve.png",

    figures_directory
    / "final_test_calibration_curve.png"
]


artifact_check = pd.DataFrame(
    [
        {
            "artifact": path.name,

            "location": str(path.parent),

            "exists": path.exists()
        }

        for path in required_artifacts
    ]
)


display(artifact_check)


missing_artifacts = artifact_check[
    artifact_check["exists"] == False
]


if missing_artifacts.empty:

    print(
        "\nAll required Notebook 5 "
        "artifacts were created."
    )

else:

    print(
        "\nThe following artifacts "
        "are missing:"
    )

    display(missing_artifacts)

In [ ]:
final_test_row = (
    test_threshold_comparison[
        test_threshold_comparison[
            "model"
        ].str.contains(
            "Tuned",
            regex=False
        )
    ]
    .iloc[0]
)


print("=" * 65)

print(
    "NOTEBOOK 5 — FINAL TUNED MODEL COMPLETE"
)

print("=" * 65)

print(
    f"Selected model: "
    f"{selected_tuned_model_name}"
)

print(
    f"Probability source: "
    f"{selected_probability_source}"
)

print(
    f"Selected threshold: "
    f"{selected_threshold:.3f}"
)

print(
    f"Test ROC-AUC: "
    f"{test_roc_auc:.4f}"
)

print(
    f"Test PR-AUC: "
    f"{test_pr_auc:.4f}"
)

print(
    f"Test precision: "
    f"{final_test_row['precision']:.4f}"
)

print(
    f"Test recall: "
    f"{final_test_row['recall']:.4f}"
)

print(
    f"Test F1: "
    f"{final_test_row['f1']:.4f}"
)

print(
    f"Test F2: "
    f"{final_test_row['f2']:.4f}"
)

print(
    f"Test Brier loss: "
    f"{test_brier_loss:.4f}"
)

print(
    "\nEvaluation model:"
)

print(
    final_evaluation_model_path
)

print(
    "\nProduction model:"
)

print(
    production_model_path
)

print(
    "\nFull OOS prediction table:"
)

print(
    model_output_directory
    / "all_customer_oos_predictions.csv"
)

print(
    "\nNext notebook:"
)

print(
    "06_model_explainability.ipynb"
)